# Structured — Retrospective Dataset — Inspection

**New dataset category (not part of the DS1–DS6 OneSec series).** Source app: *Structured* — a digital
task-planning / calendar app (Heidelberg University study, "Structured Planning as a Complex Adaptive
System"). Behavioral log = task lifecycle events (created / updated / deleted) via PostHog;
survey = Qualtrics intake (demographics, planning habits, productivity, autism/ADHD self-report, ASRS-6).

Raw files:
- `data_raw/structured/Retrospective/retro_behavioraldata_2026_07_22.csv`
- `data_raw/structured/Retrospective/retro_surveydata_2026_07_22.csv`
- `data_raw/structured/Retrospective/Retrospective_Study_Structured_Task_Created.pdf` (partial codebook — documents only the `Task Created` event schema, 3 pages)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW = Path("../data_raw/structured/Retrospective")
BEH = RAW / "retro_behavioraldata_2026_07_22.csv"
SURV = RAW / "retro_surveydata_2026_07_22.csv"
print("Paths configured")

## 1. File Inventory

In [ ]:
import os
for f in sorted(RAW.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name}  ({size_mb:.2f} MB)")

## 2. Behavioral File — Structure

No comma-separated-in-one-string issue here (unlike DS4) — standard comma-delimited CSV.

In [ ]:
beh = pd.read_csv(BEH, low_memory=False)
print("Shape:", beh.shape)
print("\nColumns:", beh.columns.tolist())
beh.head(3)

## 3. Dtypes

In [ ]:
print(beh.dtypes)

## 4. Event Types

The partial codebook PDF only documents the `Task Created` event schema (21 declared properties from the
Structured PostHog event taxonomy). The actual data contains three event classes — `Task Updated` and
`Task Deleted` are **not covered by any codebook we have**; their schema must be reverse-engineered from
the data itself.

In [ ]:
print(beh['event'].value_counts(dropna=False))

## 5. `update_type` — Mixed Encoding

`update_type` is only populated for `Task Updated` events. Values are inconsistently encoded: most look
like JSON-array strings (`["date_changed"]`, `["date_changed","start_time_changed"]`), but two values —
`subtask_completed` / `subtask_uncompleted` — are bare strings without brackets. `[]` (empty array, 1,607
rows) is a distinct value from NaN (12,968 rows) and needs separate interpretation.

In [ ]:
print(beh['update_type'].value_counts(dropna=False).head(30))
print()
print("NaN count:", beh['update_type'].isna().sum(), " — should equal Created+Deleted count:",
      (beh['event']!='Task Updated').sum())

## 6. `update_method` — Creation/Update Source Channel

In [ ]:
print(beh['update_method'].value_counts(dropna=False))

## 7. `task_type`, `subscription_type`, `recurrence_unit`

In [ ]:
for col in ['task_type', 'subscription_type', 'recurrence_unit']:
    print(f"--- {col} ---")
    print(beh[col].value_counts(dropna=False))
    print()

## 8. Timestamp Parsing — Mixed Precision

Timestamps mix fractional-second (`...152000Z`) and whole-second (`...44Z`) precision within the same
column. Naive `pd.to_datetime` inference fails on ~0.09% of rows; `format="mixed"` parses all of them
(same fix needed previously for Stanford Health HealthKit data).

In [ ]:
ts_naive = pd.to_datetime(beh['event_timestamp'], errors='coerce', utc=True)
print("Default parser failures:", ts_naive.isna().sum(), "/", len(ts_naive))

ts_mixed = pd.to_datetime(beh['event_timestamp'], format='mixed', errors='coerce', utc=True)
print("format='mixed' failures:", ts_mixed.isna().sum(), "/", len(ts_mixed))
print("Range:", ts_mixed.min(), "to", ts_mixed.max())

## 9. `event_timezone` Coverage

In [ ]:
print("Unique timezones:", beh['event_timezone'].nunique())
print(beh['event_timezone'].value_counts(dropna=False).head(15))
print()
print("Missing:", beh['event_timezone'].isna().sum())

## 10. `task_name` — Fully Empty (Privacy Discrepancy)

The Qualtrics consent text explicitly states that "task titles" are among the collected fields. In this
export, `task_name` is 100% NaN — titles were evidently stripped before export, despite the consent
wording. Flagging as a documentation/consent-text mismatch, not a data-quality defect.

In [ ]:
print("task_name non-null:", beh['task_name'].notna().sum(), "/", len(beh))

## 11. `task_start_at` / `task_end_at` — Null Structure

In [ ]:
print(pd.crosstab(beh['task_type'], beh['task_start_at'].isna()))
print()
print("task_timezone missing:", beh['task_timezone'].isna().mean()*100, "% (floating/unset scheduling tz)")

## 12. Full Null Summary — Behavioral

In [ ]:
null_pct = (beh.isnull().sum() / len(beh) * 100).round(2)
pd.DataFrame({'null_count': beh.isnull().sum(), 'null_pct': null_pct})

## 13. Participant IDs — Behavioral

In [ ]:
print("Unique user_id (behavioral):", beh['user_id'].nunique())
print("Unique task_id:", beh['task_id'].nunique())
print()
print("task_id shared across >1 user_id (should be 0):", (beh.groupby('task_id')['user_id'].nunique() > 1).sum())
print("Full duplicate rows:", beh.duplicated().sum())

## 14. Subscription Type Stability Within User

In [ ]:
sub_per_user = beh.groupby('user_id')['subscription_type'].nunique()
print("Users with >1 subscription_type over observed period:", (sub_per_user > 1).sum(), "/", len(sub_per_user))
print("→ subscription_type is time-varying for some users; treat as an event-level attribute, not a static demographic.")

## 15. Activity Span and Volume per Participant

In [ ]:
beh['ts'] = pd.to_datetime(beh['event_timestamp'], format='mixed', errors='coerce', utc=True)
span = beh.groupby('user_id').agg(n_events=('ts','size'), first=('ts','min'), last=('ts','max'))
span['days_observed'] = (span['last'] - span['first']).dt.total_seconds() / 86400
print(span[['n_events','days_observed']].describe())
print()
print("Users with <7 days span:", (span['days_observed'] < 7).sum())
print("Users with >=90 days span:", (span['days_observed'] >= 90).sum())

## 16. Events per Task (Lifecycle Depth)

In [ ]:
ev_per_task = beh.groupby('task_id').size()
print(ev_per_task.describe())
print()
print("Tasks with a single event (created, never touched again):", (ev_per_task == 1).sum())
print("Most-edited task event count:", ev_per_task.max())

## 17. Survey File — Qualtrics Header Structure

Standard 2-row Qualtrics export header: row 0 = full question text, row 1 = `ImportId` JSON. Real
response data starts at row index 2. Column names (`Q12`, `Q21`, ...) are Qualtrics internal codes with
no separate codebook document provided for this survey — the question-text row itself is the only
codebook available and must be preserved for provenance.

In [ ]:
surv_raw = pd.read_csv(SURV, low_memory=False)
print("Raw shape (incl. 2 Qualtrics header rows):", surv_raw.shape)

qtext = surv_raw.iloc[0]      # full question text
qimport = surv_raw.iloc[1]    # ImportId JSON

surv = surv_raw.iloc[2:].reset_index(drop=True)
print("Data shape after dropping header rows:", surv.shape)
surv.head(3)

## 18. Survey — Question Text Reference (reconstructed codebook)

In [ ]:
for c in surv.columns:
    if c not in ('user_id',) and not c.endswith('_TEXT'):
        print(f"{c}: {str(qtext[c])[:150]}")

## 19. Survey — Participant ID Overlap with Behavioral

In [ ]:
beh_ids = set(beh['user_id'].unique())
surv_ids = set(surv['user_id'].unique())
print("Behavioral users:", len(beh_ids))
print("Survey users:", len(surv_ids))
print("Overlap:", len(beh_ids & surv_ids))
print("Survey-only (no behavioral):", len(surv_ids - beh_ids))
print("Behavioral-only (no survey) — should be 0:", len(beh_ids - surv_ids))
print()
print("Duplicate user_id in survey:", surv['user_id'].duplicated().sum())

## 20. Survey — Consent and Completion Status

In [ ]:
print(surv['Q12'].value_counts(dropna=False))
print()
print(surv['Finished'].value_counts(dropna=False))
print()
print(surv['Progress'].value_counts(dropna=False))

## 21. Consent Compliance Check

Critical governance check: participants who explicitly declined consent, or never answered the consent
question, must not appear in the linked behavioral data.

In [ ]:
no_consent = surv[surv['Q12'] == 'I do NOT consent']
no_answer = surv[surv['Q12'].isna()]
print("Explicit non-consenters:", len(no_consent), "| present in behavioral:", len(set(no_consent['user_id']) & beh_ids))
print("No consent answer (NaN):", len(no_answer), "| present in behavioral:", len(set(no_answer['user_id']) & beh_ids))
print()
print("Result: consent gating is respected in this export." if
      (len(set(no_consent['user_id']) & beh_ids) == 0 and len(set(no_answer['user_id']) & beh_ids) == 0)
      else "WARNING: consent violation detected — investigate before proceeding.")

## 22. Survey — Likert / Percentage Value Formats

In [ ]:
for c in ['Q41','Q42','Q43','Q431','Q44','Q45','Q46']:
    print(f"--- {c}: {str(qtext[c])[:90]} ---")
    print(surv[c].value_counts(dropna=False).head(8))
    print()

## 23. Survey — ASRS-6 Screener Items (Q521_1–6)

Standard World Health Organization Adult ADHD Self-Report Scale (ASRS-v1.1) Part A wording and
5-point response set (Never / Rarely / Sometimes / Often / Very Often) — confirmed by exact string match,
not assumed. Official ASRS scoring uses per-item asymmetric thresholds for screen-positive; that
construct-level scoring is deferred to a modeling/feature-extraction stage, not applied here.

In [ ]:
for c in ['Q521_1','Q521_2','Q521_3','Q521_4','Q521_5','Q521_6']:
    print(f"--- {c} ---")
    print(surv[c].value_counts(dropna=False))
    print()

## 24. Survey — Missingness Summary

In [ ]:
null_pct = (surv.isna().mean() * 100).round(1)
pd.DataFrame({'null_pct': null_pct})

## 25. Duplicate Rows Check — Both Files

In [ ]:
print("behavioral duplicate rows:", beh.duplicated().sum())
print("survey duplicate rows (data only):", surv.duplicated().sum())

## 26. Inspection Summary

In [ ]:
print("""
=== STRUCTURED RETROSPECTIVE — INSPECTION SUMMARY ===

Behavioral (53,686 rows x 21 cols):
  Delimiter: comma (standard)
  3 event classes: Task Updated 40,718 (75.8%), Task Created 10,529 (19.6%), Task Deleted 2,439 (4.5%)
  Only "Task Created" has a codebook PDF (3 pages); Updated/Deleted schemas reverse-engineered from data
  update_type: JSON-array-like strings + 2 bare-string values (subtask_completed/uncompleted) + "[]" as
    a distinct value from NaN -- needs explicit parsing, not naive split
  event_timestamp: mixed precision (fractional vs whole-second) -- format="mixed" resolves all rows
  task_name: 100% NaN despite consent text stating titles are collected -- privacy/consent-text mismatch
  task_start_at/task_end_at: NaN exactly for all_day/inbox tasks (only "timed" tasks have them) -- MCAR
    within task_type, not a data defect
  task_timezone: 96.7% missing (most tasks use floating/unset time, unlike event_timezone which is ~complete)
  61 unique participants, 26,729 unique tasks, task_id never shared across participants
  subscription_type changes mid-study for 5/61 users -- must be treated as time-varying
  Activity span: median 132 days, 9 users <7 days, 39 users >=90 days
  Zero duplicate rows

Survey (117 real responses, after dropping 2 Qualtrics header rows from 119 total):
  Qualtrics export; only the embedded question-text row serves as a codebook (no separate codebook doc)
  61/61 behavioral participants have a survey record; 56 survey-only respondents have no behavioral data
  Consent: 73 consented, 6 explicitly declined, 38 never answered -- ALL 44 non/declined-consent users
    correctly absent from behavioral data (consent gating verified, no violation)
  Only 71/117 (60.7%) finished the survey; most incomplete responses stopped at Progress=6% (just past consent)
  Demographic/attitude items: ~40-44% missing (tracks the unfinished-response rate)
  ASRS-6 items (Q521_1-6): confirmed standard ASRS-v1.1 wording/response set; 54.7% missing (extra skip logic
    beyond base non-completion, not yet explained by any codebook)
  Likert items (Q41-Q46) encode both label and numeric grade in one string (e.g. "6 - happy") -- extractable
  Percentage items (Q43, Q431) are strings with a "%" suffix -- need numeric coercion
""")